In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
warnings.filterwarnings('ignore')



In [11]:
#Read data 
df = pd.read_csv("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\2025_masters_tournament_r1_values.csv")
df.head()

,position,player_name,total_score,r1_score,sg_putt,sg_arg,sg_app,sg_ott,sg_t2g,sg_total,distance,accuracy,gir,prox_fw
0,1,"McIlroy, Rory",-11,0,-1.108,-0.932,2.045,1.584,2.697,1.589,318.5,0.71429,0.61111,31.0810
1,2,"Rose, Justin",-11,-7,5.270,1.643,2.785,-1.108,3.320,8.589,293.6,0.64286,0.77778,26.4087
2,3,"Reed, Patrick",-9,-1,-0.623,0.147,2.018,1.047,3.212,2.589,297.2,0.92857,0.72222,34.6933
3,4,"Scheffler, Scottie",-8,-4,2.902,1.163,0.537,0.988,2.688,5.589,306.7,0.71429,0.61111,42.4736
4,T5,"Im, Sungjae",-7,-1,2.604,0.111,-0.097,-0.028,-0.014,2.589,302.4,0.64286,0.55556,26.4601


In [12]:
df = pd.read_csv("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\2025_masters_tournament_r2_values.csv")
df.head()

,position,player_name,total_score,r2_score,sg_putt,sg_arg,sg_app,sg_ott,sg_t2g,sg_total,distance,accuracy,gir,prox_fw
0,1,"McIlroy, Rory",-11,-6,2.151,0.413,3.643,0.467,4.523,6.674,316.0,0.50000,0.77778,32.9339
1,2,"Rose, Justin",-11,-1,2.082,0.020,-1.043,0.615,-0.408,1.674,289.2,0.85714,0.55556,66.6944
2,3,"Reed, Patrick",-9,-2,0.113,3.286,-1.178,0.453,2.561,2.674,296.2,0.85714,0.61111,52.2363
3,4,"Scheffler, Scottie",-8,-1,-0.958,1.878,1.319,-0.566,2.632,1.674,299.9,0.71429,0.66667,30.2704
4,T5,"Im, Sungjae",-7,-2,1.463,1.047,-0.113,0.276,1.210,2.674,298.2,0.78571,0.77778,39.6705


Creating a New dataset where the two rounds are averaged out 

In [13]:
r1 = pd.read_csv("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\2025_masters_tournament_r1_values.csv")
r2 = pd.read_csv("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\2025_masters_tournament_r2_values.csv")

merged = pd.merge(r1, r2, on="player_name", suffixes=("_r1", "_r2"))

avg_df = pd.DataFrame()
avg_df["player_name"] = merged["player_name"]

avg_df["position"] = merged["position_r2"]

common_cols = [
    "total_score",
    "sg_putt",
    "sg_arg",
    "sg_app",
    "sg_ott",
    "sg_t2g",
    "sg_total",
    "distance",
    "accuracy",
    "gir",
    "prox_fw"
]
for col in common_cols:
    avg_df[col] = (merged[f"{col}_r1"] + merged[f"{col}_r2"]) / 2

avg_df["avg_round_score"] = (merged["r1_score"] + merged["r2_score"]) / 2

print(avg_df.head())

          player_name position  total_score  sg_putt  sg_arg  sg_app  sg_ott  \
0       McIlroy, Rory        1        -11.0   0.5215 -0.2595   2.844  1.0255   
1        Rose, Justin        2        -11.0   3.6760  0.8315   0.871 -0.2465   
2       Reed, Patrick        3         -9.0  -0.2550  1.7165   0.420  0.7500   
3  Scheffler, Scottie        4         -8.0   0.9720  1.5205   0.928  0.2110   
4         Im, Sungjae       T5         -7.0   2.0335  0.5790  -0.105  0.1240   

   sg_t2g  sg_total  distance  accuracy       gir   prox_fw  avg_round_score  
0  3.6100    4.1315    317.25  0.607145  0.694445  32.00745             -3.0  
1  1.4560    5.1315    291.40  0.750000  0.666670  46.55155             -4.0  
2  2.8865    2.6315    296.70  0.892855  0.666665  43.46480             -1.5  
3  2.6600    3.6315    303.30  0.714290  0.638890  36.37200             -2.5  
4  0.5980    2.6315    300.30  0.714285  0.666670  33.06530             -1.5  


In [15]:
df = pd.read_csv("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\masters_avg_dataset.csv")
df.head()

,player_name,total_score,sg_putt,sg_arg,sg_app,sg_ott,sg_t2g,sg_total,distance,accuracy,gir,prox_fw,avg_round_score
0,"McIlroy, Rory",-11.0,0.5215,-0.2595,2.844,1.0255,3.6100,4.1315,317.25,0.607145,0.694445,32.00745,-3.0
1,"Rose, Justin",-11.0,3.6760,0.8315,0.871,-0.2465,1.4560,5.1315,291.40,0.750000,0.666670,46.55155,-4.0
2,"Reed, Patrick",-9.0,-0.2550,1.7165,0.420,0.7500,2.8865,2.6315,296.70,0.892855,0.666665,43.46480,-1.5
3,"Scheffler, Scottie",-8.0,0.9720,1.5205,0.928,0.2110,2.6600,3.6315,303.30,0.714290,0.638890,36.37200,-2.5
4,"Im, Sungjae",-7.0,2.0335,0.5790,-0.105,0.1240,0.5980,2.6315,300.30,0.714285,0.666670,33.06530,-1.5


Basic Model

In [27]:


def load_data(filepath: str) -> pd.DataFrame:
   return pd.read_csv(filepath)


def create_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(by="total_score")  # lower score = better
    
    df["made_cut"] = 0
    df.iloc[:50, df.columns.get_loc("made_cut")] = 1
    
    return df


def prepare_features(df: pd.DataFrame):
   features = [
        "sg_putt", "sg_arg", "sg_app", "sg_ott",
        "sg_t2g", "sg_total",
        "distance", "accuracy", "gir", "prox_fw",
        "avg_round_score"
    ]
   X = df[features]
   y = df["made_cut"]
   return X, y


def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    return model, y_test, y_pred


def evaluate_model(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


def main():
    df = load_data("C:\\Users\\sawye\\Downloads\\DataScience(1)\\DATASET\\masters_avg_dataset.csv")
    
    df = create_target(df)
    
    X, y = prepare_features(df)
    
    model, y_test, y_pred = train_model(X, y)
    
    evaluate_model(y_test, y_pred)


if __name__ == "__main__":
    main()
    
    df = create_target(df)
    
    X, y = prepare_features(df)
    
    model, y_test, y_pred = train_model(X, y)
    
    evaluate_model(y_test, y_pred)







Accuracy: 0.631578947368421

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.67      0.53         6
           1       0.80      0.62      0.70        13

    accuracy                           0.63        19
   macro avg       0.62      0.64      0.61        19
weighted avg       0.69      0.63      0.64        19

Accuracy: 0.631578947368421

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.67      0.53         6
           1       0.80      0.62      0.70        13

    accuracy                           0.63        19
   macro avg       0.62      0.64      0.61        19
weighted avg       0.69      0.63      0.64        19



In [28]:
def find_minimum_profile(model, X):
    """
    Finds approximate minimum stats needed to make the cut
    using average feature values and model coefficients
    """

    # Get average player stats
    avg_values = X.mean()

    # Get model coefficients
    coefs = model.coef_[0]
    intercept = model.intercept_[0]

    print("\n--- MINIMUM PROFILE TO MAKE CUT (Approx) ---\n")

    # Solve for probability = 0.5
    # Logistic equation: z = 0 → boundary
    # z = intercept + sum(coef * x)

    # We'll isolate one important variable (sg_total)
    feature_names = X.columns.tolist()

    # Choose most important feature (largest coefficient)
    important_idx = np.argmax(np.abs(coefs))
    important_feature = feature_names[important_idx]

    print(f"Most important stat: {important_feature}")

    # Solve for that feature while holding others constant
    other_sum = intercept
    for i, f in enumerate(feature_names):
        if i != important_idx:
            other_sum += coefs[i] * avg_values[f]

    required_value = -other_sum / coefs[important_idx]

    print(f"\nEstimated minimum {important_feature}: {required_value:.3f}")

    print("\nOther stats held at average:")
    print(avg_values)

In [29]:
find_minimum_profile(model, X)


--- MINIMUM PROFILE TO MAKE CUT (Approx) ---

Most important stat: sg_arg

Estimated minimum sg_arg: 0.142

Other stats held at average:
sg_putt             -0.000005
sg_arg              -0.000016
sg_app              -0.000016
sg_ott              -0.000053
sg_t2g              -0.000005
sg_total            -0.000079
distance           298.066316
accuracy             0.734211
gir                  0.619884
prox_fw             38.440987
avg_round_score      1.131579
dtype: float64


In [30]:
prob = model.predict_proba(X)[:, 1]

cut_players = X[prob >= 0.7]

print("\n--- AVERAGE STATS OF SAFE PLAYERS ---\n")
print(cut_players.mean())


--- AVERAGE STATS OF SAFE PLAYERS ---

sg_putt              0.572816
sg_arg               0.563132
sg_app               0.905697
sg_ott               0.208316
sg_t2g               1.677145
sg_total             2.249921
distance           300.410526
accuracy             0.740602
gir                  0.674708
prox_fw             36.662524
avg_round_score     -1.118421
dtype: float64
